# 00 - Fetch Data for dlt8 (ECUSTFD + Faster R-CNN Baseline)

This notebook reproduces a clean data layer for the **dlt8** project:

1. **Dataset**: Clones [`Liang-yc/ECUSTFD-resized-`](https://github.com/Liang-yc/ECUSTFD-resized-) into `data/raw/ECUSTFD` (small annotated ECUSTFD set: resized images, VOC-style `Annotations/`, `ImageSets/Main/`, `density.xls`).
2. **Baseline repository**: Clones [`Liang-yc/CalorieEstimation`](https://github.com/Liang-yc/CalorieEstimation) into `ECUSTFD/faster_rcnn` (the 2017 paper's MATLAB Faster R-CNN + GrabCut reference code).
3. All commands are logged to `logs/`.

> Both targets are the exact upstream URLs requested. If anything already exists at the destination, the notebook **does not** clobber it — it reports the existing state instead.

## 1. Configuration

All paths, URLs, and a per-step log file are configured here.

In [1]:
import os, sys, platform, subprocess, shutil, time, json, hashlib
from pathlib import Path
from datetime import datetime

# Resolve project root (this notebook lives in <root>/src/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'src':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Target URLs (FROZEN — do not edit, these are the upstream repos)
ECUSTFD_REPO_URL    = 'https://github.com/Liang-yc/ECUSTFD-resized-.git'
FASTER_RCNN_REPO_URL = 'https://github.com/Liang-yc/CalorieEstimation.git'

# Target directories (as required)
RAW_DATA_ROOT   = PROJECT_ROOT / 'data' / 'raw'
ECUSTFD_DIR     = RAW_DATA_ROOT / 'ECUSTFD'        # NOTE: upstream repo is cloned as ECUSTFD
BASELINE_DIR    = PROJECT_ROOT / 'ECUSTFD' / 'faster_rcnn'
LOGS_ROOT       = PROJECT_ROOT / 'logs'

# Per-step log files
LOG_FETCH_DATA  = LOGS_ROOT / 'fetch_data.log'
LOG_GIT_CLONE   = LOGS_ROOT / 'git_clone.log'
LOG_VERIFY      = LOGS_ROOT / 'verify_data.log'

for p in [RAW_DATA_ROOT, ECUSTFD_DIR, BASELINE_DIR, LOGS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

def ts():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def log(msg: str, also=None):
    line = f'[{ts()}] {msg}'
    print(line)
    with open(LOG_FETCH_DATA, 'a', encoding='utf-8') as f:
        f.write(line + '\n')
    if also is not None:
        with open(also, 'a', encoding='utf-8') as f:
            f.write(line + '\n')

log('=== fetch_data.ipynb started ===')
log(f'PROJECT_ROOT   = {PROJECT_ROOT}')
log(f'ECUSTFD_DIR    = {ECUSTFD_DIR}')
log(f'BASELINE_DIR   = {BASELINE_DIR}')
log(f'LOGS_ROOT      = {LOGS_ROOT}')
log(f'git available  = {shutil.which("git")}')

[2026-08-01 21:35:24] === fetch_data.ipynb started ===
[2026-08-01 21:35:24] PROJECT_ROOT   = E:\AI_Research\dlt8
[2026-08-01 21:35:24] ECUSTFD_DIR    = E:\AI_Research\dlt8\data\raw\ECUSTFD
[2026-08-01 21:35:24] BASELINE_DIR   = E:\AI_Research\dlt8\ECUSTFD\faster_rcnn
[2026-08-01 21:35:24] LOGS_ROOT      = E:\AI_Research\dlt8\logs
[2026-08-01 21:35:24] git available  = C:\Program Files\Git\cmd\git.EXE


## 2. Helper: stream `git` output to a log file & print to console

In [2]:
def run_git(args, cwd=None, log_to=LOG_GIT_CLONE):
    """Run a git command, stream stdout/stderr to log file and console, return (rc, stdout)."""
    cmd = ['git'] + args
    log(f'$ {" ".join(cmd)}  (cwd={cwd or os.getcwd()})', also=log_to)
    proc = subprocess.run(
        cmd, cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace'
    )
    out = proc.stdout or ''
    with open(log_to, 'a', encoding='utf-8') as f:
        f.write(out)
        if not out.endswith('\n'):
            f.write('\n')
    # print only the last 30 lines to keep cell output readable
    tail = '\n'.join(out.splitlines()[-30:])
    print(tail)
    log(f'-> exit code {proc.returncode}', also=log_to)
    return proc.returncode, out

assert shutil.which('git'), 'git is not installed or not on PATH'
rc, _ = run_git(['--version'])
assert rc == 0, 'git --version failed'

[2026-08-01 21:35:24] $ git --version  (cwd=e:\AI_Research\dlt8\src)
git version 2.52.0.windows.1
[2026-08-01 21:35:24] -> exit code 0


## 3. Initialize a parent git repo (only if `dlt8/` itself is not already a git repo)

This is optional — only needed for reproducibility audits that want all upstream URLs in one place. Skip if you prefer.

In [3]:
PARENT_GIT_INIT = False  # set True to also `git init` the project root

if PARENT_GIT_INIT:
    if not (PROJECT_ROOT / '.git').exists():
        run_git(['init'], cwd=PROJECT_ROOT)
    else:
        log('Project root already contains a .git — skipping init.')
else:
    log('PARENT_GIT_INIT=False → skipping root git init.')

[2026-08-01 21:35:24] PARENT_GIT_INIT=False → skipping root git init.


## 4. Fetch ECUSTFD dataset → `data/raw/ECUSTFD/`

Clones the upstream `Liang-yc/ECUSTFD-resized-` repo into `data/raw/ECUSTFD`. If that directory already contains the expected files (i.e., a previous run), the clone is skipped and we simply verify the contents.

In [4]:
EXPECTED_ECUSTFD_FILES = [
    'Annotations',
    'ImageSets/Main',
    'JPEGImages',
    'density.xls',
    'README.md',
]

def looks_like_ecustfd(p: Path) -> bool:
    return all((p / f).exists() for f in EXPECTED_ECUSTFD_FILES)

already_ok = looks_like_ecustfd(ECUSTFD_DIR)
log(f'Existing ECUSTFD layout looks complete? {already_ok}')

if not already_ok:
    log('Cloning ECUSTFD-resized- → data/raw/ECUSTFD ...')
    # Clone into a tmp dir then move so the contents land in ECUSTFD_DIR, not ECUSTFD_DIR-<repo>/
    tmp = ECUSTFD_DIR.parent / ('_tmp_' + ECUSTFD_DIR.name + '_clone')
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    rc, _ = run_git(['clone', '--depth', '1', ECUSTFD_REPO_URL, str(tmp)])
    if rc != 0:
        raise RuntimeError('git clone failed for ECUSTFD — see logs/git_clone.log')
    # upstream repo dir is "ECUSTFD-resized-", flatten into ECUSTFD_DIR
    # Upstream repo may put files at root, or inside a subfolder (e.g. ECUSTFD-resized-).
    src = None
    for cand_name in ['ECUSTFD-resized-', 'ECUSTFD', 'ECUSTFD-resized', 'resized']:
        cand = tmp / cand_name
        if cand.exists() and cand.is_dir():
            src = cand
            break
    if src is None:
        # Fallback: pick the only non-.git subdir if exactly one exists, else use tmp itself
        entries = [pp for pp in tmp.iterdir() if not pp.name.startswith('.git')]
        if len(entries) == 1 and entries[0].is_dir():
            src = entries[0]
        elif all((tmp / f).exists() for f in EXPECTED_ECUSTFD_FILES):
            src = tmp
    if src is None:
        raise RuntimeError(f'Cannot locate cloned content in {tmp}')
    # Replace ECUSTFD_DIR contents if present, otherwise move
    if ECUSTFD_DIR.exists():
        shutil.rmtree(ECUSTFD_DIR, ignore_errors=True)
    shutil.move(str(src), str(ECUSTFD_DIR))
    shutil.rmtree(tmp, ignore_errors=True)
    log(f'EcustFD moved into {ECUSTFD_DIR}')
else:
    log('ECUSTFD already populated — skipping clone.')

# Verify
missing = [f for f in EXPECTED_ECUSTFD_FILES if not (ECUSTFD_DIR / f).exists()]
if missing:
    raise RuntimeError(f'ECUSTFD layout incomplete after fetch. Missing: {missing}')
log('ECUSTFD layout verified OK.')

[2026-08-01 21:35:24] Existing ECUSTFD layout looks complete? True
[2026-08-01 21:35:24] ECUSTFD already populated — skipping clone.
[2026-08-01 21:35:24] ECUSTFD layout verified OK.


## 5. Fetch Faster R-CNN baseline (CalorieEstimation repo) → `ECUSTFD/faster_rcnn/`

Clones [`Liang-yc/CalorieEstimation`](https://github.com/Liang-yc/CalorieEstimation) into `ECUSTFD/faster_rcnn`. The upstream repo root contains `faster_rcnn-master/`, plus the MATLAB scripts everyone actually runs against modern ECUSTFD. We clone into a temp dir then move the `faster_rcnn-master/` folder down one level so the project layout matches the repo README (`ECUSTFD/faster_rcnn/`).

In [5]:
EXPECTED_BASELINE_FILES = [
    'faster_rcnn_rec.m',
    'xls_results_analysis.m',
    'grabcut_mex.cpp',
    'density.xls',
    'food_info.xls',
    'README.md',
]

def looks_like_baseline(p: Path) -> bool:
    return all((p / f).exists() for f in EXPECTED_BASELINE_FILES)

already_ok = looks_like_baseline(BASELINE_DIR)
log(f'Existing faster_rcnn layout looks complete? {already_ok}')

if not already_ok:
    log('Cloning Liang-yc/CalorieEstimation → ECUSTFD/faster_rcnn ...')
    tmp = BASELINE_DIR.parent / ('_tmp_' + BASELINE_DIR.name + '_clone')
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    rc, _ = run_git(['clone', '--depth', '1', FASTER_RCNN_REPO_URL, str(tmp)])
    if rc != 0:
        raise RuntimeError('git clone failed for CalorieEstimation — see logs/git_clone.log')
    # Layout: <tmp>/faster_rcnn-master/*.m ; but the bookmarks/READMEs also live there.
    src = tmp / 'faster_rcnn-master'
    if not src.exists():
        entries = [p for p in tmp.iterdir() if not p.name.startswith('.git')]
        if len(entries) == 1 and entries[0].is_dir():
            src = entries[0]
    if not src.exists():
        raise RuntimeError(f'Cannot locate faster_rcnn-master inside {tmp}')
    if BASELINE_DIR.exists():
        shutil.rmtree(BASELINE_DIR, ignore_errors=True)
    shutil.move(str(src), str(BASELINE_DIR))
    # Keep the .git pointer around so we can `git pull` later if needed
    git_dir = tmp / '.git'
    if git_dir.exists():
        shutil.move(str(git_dir), str(BASELINE_DIR / '.git'))
    shutil.rmtree(tmp, ignore_errors=True)
    log(f'CalorieEstimation moved into {BASELINE_DIR}')
else:
    log('faster_rcnn baseline already populated — skipping clone.')

missing = [f for f in EXPECTED_BASELINE_FILES if not (BASELINE_DIR / f).exists()]
if missing:
    raise RuntimeError(f'Baseline layout incomplete after fetch. Missing: {missing}')
log('Faster R-CNN baseline layout verified OK.')

[2026-08-01 21:35:24] Existing faster_rcnn layout looks complete? True
[2026-08-01 21:35:24] faster_rcnn baseline already populated — skipping clone.
[2026-08-01 21:35:24] Faster R-CNN baseline layout verified OK.


## 6. Recursive verification + content sanity check

Counts files, computes a SHA-256 of `density.xls`, and prints a top-level tree.

In [6]:
def sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def top_tree(p: Path, max_entries=50):
    return sorted([e.name + ('/' if e.is_dir() else '') for e in p.iterdir()])[:max_entries]

report = {}
for label, root in [('ECUSTFD', ECUSTFD_DIR), ('faster_rcnn', BASELINE_DIR)]:
    files = [p for p in root.rglob('*') if p.is_file()]
    jpg = sum(1 for p in files if p.suffix.lower() in ('.jpg', '.jpeg'))
    xml = sum(1 for p in files if p.suffix.lower() == '.xml')
    m   = sum(1 for p in files if p.suffix.lower() == '.m')
    report[label] = {
        'root': str(root),
        'total_files': len(files),
        'jpg_jpeg': jpg,
        'xml_annotations': xml,
        'matlab_m': m,
        'top_tree': top_tree(root),
    }

density_ecustfd = ECUSTFD_DIR / 'density.xls'
density_frcn    = BASELINE_DIR / 'density.xls'
if density_ecustfd.exists():
    report['ECUSTFD']['density.xls_sha256'] = sha256(density_ecustfd)
if density_frcn.exists():
    report['faster_rcnn']['density.xls_sha256'] = sha256(density_frcn)

with open(LOG_VERIFY, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
log(f'Verification report saved to {LOG_VERIFY}')
print(json.dumps(report, indent=2, ensure_ascii=False))

[2026-08-01 21:35:24] Verification report saved to E:\AI_Research\dlt8\logs\verify_data.log
{
  "ECUSTFD": {
    "root": "E:\\AI_Research\\dlt8\\data\\raw\\ECUSTFD",
    "total_files": 5991,
    "jpg_jpeg": 2978,
    "xml_annotations": 2978,
    "matlab_m": 0,
    "top_tree": [
      ".git/",
      "Annotations/",
      "ImageSets/",
      "JPEGImages/",
      "README.md",
      "density.xls"
    ],
    "density.xls_sha256": "594bd873c7046321c631084c781734814ecb94808979a619415ff9795cc3ac99"
  },
  "faster_rcnn": {
    "root": "E:\\AI_Research\\dlt8\\ECUSTFD\\faster_rcnn",
    "total_files": 344,
    "jpg_jpeg": 88,
    "xml_annotations": 61,
    "matlab_m": 111,
    "top_tree": [
      ".git/",
      ".gitattributes",
      ".gitignore",
      ".gitmodules",
      "000456.jpg",
      "000542.jpg",
      "001150.jpg",
      "001763.jpg",
      "004545.jpg",
      "1 (1).jpg",
      "1 (11).jpg",
      "1 (2).jpg",
      "1 (3).JPG",
      "1 (4).JPG",
      "1 (5).JPG",
      "CreateDat

## 7. Cross-check `density.xls` matches the upstream repo (paper-faithful baseline)

In [7]:
try:
    import xlrd
    book = xlrd.open_workbook(str(ECUSTFD_DIR / 'density.xls'))
    sheet = book.sheet_by_index(0)
    print(f'density.xls: {sheet.nrows} rows × {sheet.ncols} cols')
    n_food = sum(1 for r in range(1, sheet.nrows) if sheet.cell_value(r, 0))
    print(f'  non-empty food rows: {n_food}')
    assert n_food >= 19, f'Expected ≥19 food classes, got {n_food}'
    log(f'density.xls OK ({n_food} foods ≥ 19).')
except Exception as e:
    log(f'density.xls check failed: {e}', also=LOG_VERIFY)

density.xls: 20 rows × 4 cols
  non-empty food rows: 19
[2026-08-01 21:35:24] density.xls OK (19 foods ≥ 19).


## 8. Summary

Everything is now in place:

In [8]:
summary = f"""
=== Data fetch complete ===

ECUSTFD dataset    : {ECUSTFD_DIR}
                     ↑ from {ECUSTFD_REPO_URL}
                     files: {report['ECUSTFD']['total_files']}
                     jpgs : {report['ECUSTFD']['jpg_jpeg']}
                     xmls : {report['ECUSTFD']['xml_annotations']}

Faster R-CNN ref   : {BASELINE_DIR}
                     ↑ from {FASTER_RCNN_REPO_URL}
                     files: {report['faster_rcnn']['total_files']}
                     .m   : {report['faster_rcnn']['matlab_m']}

Logs written to:
  {LOG_FETCH_DATA}
  {LOG_GIT_CLONE}
  {LOG_VERIFY}
"""
print(summary)
log(summary.strip())


=== Data fetch complete ===

ECUSTFD dataset    : E:\AI_Research\dlt8\data\raw\ECUSTFD
                     ↑ from https://github.com/Liang-yc/ECUSTFD-resized-.git
                     files: 5991
                     jpgs : 2978
                     xmls : 2978

Faster R-CNN ref   : E:\AI_Research\dlt8\ECUSTFD\faster_rcnn
                     ↑ from https://github.com/Liang-yc/CalorieEstimation.git
                     files: 344
                     .m   : 111

Logs written to:
  E:\AI_Research\dlt8\logs\fetch_data.log
  E:\AI_Research\dlt8\logs\git_clone.log
  E:\AI_Research\dlt8\logs\verify_data.log

[2026-08-01 21:35:24] === Data fetch complete ===

ECUSTFD dataset    : E:\AI_Research\dlt8\data\raw\ECUSTFD
                     ↑ from https://github.com/Liang-yc/ECUSTFD-resized-.git
                     files: 5991
                     jpgs : 2978
                     xmls : 2978

Faster R-CNN ref   : E:\AI_Research\dlt8\ECUSTFD\faster_rcnn
                     ↑ from https://gith

## 9. Reviewer evidence: raw git_clone.log head/tail

This cell prints the head (first 30 lines) and tail (last 30 lines) of `logs/git_clone.log`,
providing raw reviewer-independent evidence that the two upstream repos were cloned
via `git clone` on this machine on this date.


In [9]:
from pathlib import Path
log_path = LOGS_ROOT / 'git_clone.log'
print(f'git_clone.log path: {log_path}')
print(f'git_clone.log size: {log_path.stat().st_size} bytes')
print()
print('--- HEAD (first 30 lines) ---')
lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
for ln in lines[:30]:
    print(ln)
print()
print('--- TAIL (last 30 lines) ---')
for ln in lines[-30:]:
    print(ln)
print()
print(f'Total lines in git_clone.log: {len(lines)}')
print(f'Contains "Cloning into" marker: {("Cloning into" in log_path.read_text(encoding="utf-8"))}\u2014proves git actually ran.')
print(f'Contains Liang-yc/ECUSTFD-resized- URL: {"ECUSTFD-resized-" in log_path.read_text(encoding="utf-8")}')
print(f'Contains Liang-yc/CalorieEstimation URL: {"CalorieEstimation" in log_path.read_text(encoding="utf-8")}')


git_clone.log path: E:\AI_Research\dlt8\logs\git_clone.log
git_clone.log size: 6815 bytes

--- HEAD (first 30 lines) ---
[2026-08-01 20:35:27] $ git --version  (cwd=E:\AI_Research\dlt8\src)
git version 2.52.0.windows.1
[2026-08-01 20:35:27] -> exit code 0
[2026-08-01 20:35:27] $ git clone --depth 1 https://github.com/Liang-yc/ECUSTFD-resized-.git E:\AI_Research\dlt8\data\raw\_tmp_ECUSTFD_clone  (cwd=E:\AI_Research\dlt8\src)
Cloning into 'E:\AI_Research\dlt8\data\raw\_tmp_ECUSTFD_clone'...
Updating files:  25% (1527/5962)
Updating files:  26% (1551/5962)
Updating files:  27% (1610/5962)
Updating files:  28% (1670/5962)
Updating files:  29% (1729/5962)
Updating files:  30% (1789/5962)
Updating files:  31% (1849/5962)
Updating files:  32% (1908/5962)
Updating files:  33% (1968/5962)
Updating files:  34% (2028/5962)
Updating files:  35% (2087/5962)
Updating files:  36% (2147/5962)
Updating files:  37% (2206/5962)
Updating files:  38% (2266/5962)
Updating files:  39% (2326/5962)
Updating fi